# STL-BERT Baseline — Google Colab

**Before running:**
1. Set runtime to GPU: `Runtime > Change runtime type > T4 GPU`
2. Upload your data to Google Drive at: `MyDrive/mtl-bert/data/`
   - `data/sarcasm/sarcasm.csv`
   - `data/cyberbullying/cyberbullying.csv`
   - `data/emotions/emotions.csv`
3. Run all cells in order.

Checkpoints and results are saved to Drive so they survive session disconnects.

In [ ]:
# Install dependencies (transformers not pre-installed on all Colab versions)
!pip install -q transformers scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Change this if your Drive folder is named differently ──
BASE_DIR      = "/content/drive/MyDrive/mtl-bert"
DATA_DIR      = os.path.join(BASE_DIR, "data")
CKPT_DIR      = os.path.join(BASE_DIR, "checkpoints", "stl-bert")
RESULTS_DIR   = os.path.join(BASE_DIR, "results", "stl-bert")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Base dir : {BASE_DIR}")
print(f"Data dir : {DATA_DIR}")
print(f"Results  : {RESULTS_DIR}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import random
import json
import csv
from typing import List, Tuple

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Dataset helpers (inlined from dataset.py) ──────────────────────────

def create_sample_datasets():
    datasets = {}

    # Sarcasm — CSV: id,class,text
    sarc_path = os.path.join(DATA_DIR, "sarcasm", "sarcasm.csv")
    sarc_data = []
    with open(sarc_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 3:
                try:
                    label = int(row[1])
                    text  = row[2].strip()
                    if text:
                        sarc_data.append((text, label))
                except ValueError:
                    continue
    datasets["sarc"] = sarc_data

    # Cyberbullying — CSV: id,class,text
    cyber_path = os.path.join(DATA_DIR, "cyberbullying", "cyberbullying.csv")
    intent_data = []
    with open(cyber_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 3:
                try:
                    label = int(row[1])
                    text  = row[2].strip()
                    if text:
                        intent_data.append((text, label))
                except ValueError:
                    continue
    datasets["intent"] = intent_data

    # Emotions — CSV: class,text
    emo_path = os.path.join(DATA_DIR, "emotions", "emotions.csv")
    emotion_data = []
    with open(emo_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 2:
                try:
                    label = int(row[0])
                    text  = row[1].strip()
                    if text:
                        emotion_data.append((text, label))
                except ValueError:
                    continue
    datasets["emotion"] = emotion_data

    return datasets


def compute_metrics(predictions, labels):
    return {
        "accuracy" : accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall"   : recall_score(labels, predictions, average="weighted", zero_division=0),
        "f1"       : f1_score(labels, predictions, average="weighted", zero_division=0),
    }


def compute_per_class_metrics(predictions, labels, class_names=None):
    report_str  = classification_report(labels, predictions, target_names=class_names, zero_division=0)
    report_dict = classification_report(labels, predictions, target_names=class_names, zero_division=0, output_dict=True)
    cm = confusion_matrix(labels, predictions)
    return {"report_str": report_str, "report_dict": report_dict, "confusion_matrix": cm.tolist()}

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ── SingleTaskDataset ──────────────────────────────────────────────────

class SingleTaskDataset(Dataset):
    def __init__(self, data: List[Tuple], tokenizer, max_length=128):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, label = self.samples[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(label, dtype=torch.long)
        }


# ── SingleTaskBERT ─────────────────────────────────────────────────────

class SingleTaskBERT(nn.Module):
    def __init__(self, model_name: str, num_classes: int):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.num_classes = num_classes
        if num_classes == 2:
            self.classifier = nn.Linear(hidden_size, 1)
        else:
            self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled       = outputs.last_hidden_state[:, 0]  # [CLS]
        return self.classifier(pooled)


# ── Evaluation ─────────────────────────────────────────────────────────

def evaluate_model(model, dataloader, num_classes, dev):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(dev)
            attention_mask = batch['attention_mask'].to(dev)
            labels         = batch['label']
            logits         = model(input_ids, attention_mask)
            if num_classes == 2:
                preds = (torch.sigmoid(logits.squeeze(-1)) > 0.5).long()
            else:
                preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    metrics = compute_metrics(all_preds, all_labels)
    return metrics, all_preds, all_labels


# ── Training function ──────────────────────────────────────────────────

def train_single_task(task_name, num_classes, train_data, val_data, test_data,
                      tokenizer, model_name, dev, batch_size, num_epochs, max_length, seed=0):
    ckpt_path = os.path.join(CKPT_DIR, f"{task_name}_seed{seed}.pt")

    train_ds = SingleTaskDataset(train_data, tokenizer, max_length)
    val_ds   = SingleTaskDataset(val_data,   tokenizer, max_length)
    test_ds  = SingleTaskDataset(test_data,  tokenizer, max_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    model     = SingleTaskBERT(model_name, num_classes).to(dev)
    optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    loss_fn   = nn.BCEWithLogitsLoss() if num_classes == 2 else nn.CrossEntropyLoss()

    best_val_f1      = 0.0
    best_model_state = None
    start_epoch      = 0

    # Resume from checkpoint
    if os.path.exists(ckpt_path):
        print(f"  Resuming from checkpoint: {ckpt_path}")
        try:
            ckpt = torch.load(ckpt_path, map_location=dev, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            best_val_f1      = ckpt['best_val_f1']
            best_model_state = ckpt.get('best_model_state')
            if ckpt.get('mid_epoch', False):
                start_epoch = ckpt['epoch']
                print(f"  Resumed mid-epoch at epoch {start_epoch+1}, best Val F1: {best_val_f1:.4f}")
            else:
                start_epoch = ckpt['epoch'] + 1
                print(f"  Resumed from epoch {start_epoch}, best Val F1: {best_val_f1:.4f}")
        except Exception as e:
            print(f"  WARNING: Checkpoint corrupted ({e}). Starting from scratch.")
            os.remove(ckpt_path)

    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_loss  = 0.0
        num_batches = 0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(dev)
            attention_mask = batch['attention_mask'].to(dev)
            labels         = batch['label'].to(dev)
            logits         = model(input_ids, attention_mask)

            if num_classes == 2:
                loss = loss_fn(logits.squeeze(-1), labels.float())
            else:
                loss = loss_fn(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss  += loss.item()
            num_batches += 1

            if num_batches % 100 == 0:
                print(f"    Step {num_batches}/{len(train_loader)}, Loss: {loss.item():.4f}")

            # Mid-epoch checkpoint every 1000 steps
            if num_batches % 1000 == 0:
                torch.save({
                    'epoch': epoch, 'mid_epoch': True,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_val_f1': best_val_f1,
                    'best_model_state': best_model_state,
                }, ckpt_path)
                print(f"    Mid-epoch checkpoint saved (step {num_batches})")

        avg_loss = total_loss / max(num_batches, 1)

        # Validate
        val_metrics, _, _ = evaluate_model(model, val_loader, num_classes, dev)
        print(f"  Epoch {epoch+1}/{num_epochs} - "
              f"Loss: {avg_loss:.4f} - "
              f"Val Acc: {val_metrics['accuracy']:.4f}, "
              f"Val F1: {val_metrics['f1']:.4f}")

        # Best model selection by validation F1
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Save checkpoint
        torch.save({
            'epoch': epoch, 'mid_epoch': False,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_f1': best_val_f1,
            'best_model_state': best_model_state,
        }, ckpt_path)
        print(f"  Checkpoint saved: {ckpt_path}")

    # Load best model for test evaluation
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    model.to(dev)
    test_metrics, all_preds, all_labels = evaluate_model(model, test_loader, num_classes, dev)

    # Clean up checkpoint
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)

    return test_metrics, all_preds, all_labels


def load_json(path):
    if not os.path.exists(path):
        return None
    with open(path, 'r') as f:
        return json.load(f)

In [ ]:
# ── Main training ──────────────────────────────────────────────────────

print("=" * 60)
print("  Single-Task BERT Baseline - Multi-Seed Training")
print("=" * 60)

MODEL_NAME      = "bert-base-uncased"
SEEDS           = [42, 123, 456]
BATCH_SIZE      = 16
NUM_EPOCHS      = 5
MAX_LENGTH      = 128
EMOTION_CLASSES = ["sad", "joy", "love", "angry", "fear", "surprise"]

task_configs = {'sarc': 2, 'intent': 2, 'emotion': 6}

print(f"\nLoading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nLoading datasets...")
tasks_data = create_sample_datasets()
for task_name, data in tasks_data.items():
    print(f"  {task_name}: {len(data)} samples")

# Resume support
progress_path       = os.path.join(RESULTS_DIR, "training_progress.json")
all_seed_results     = {t: [] for t in task_configs}
all_seed_predictions = {t: [] for t in task_configs}
all_seed_labels      = {t: [] for t in task_configs}
completed            = set()

if os.path.exists(progress_path):
    progress = load_json(progress_path)
    if progress:
        completed            = set(progress.get("completed", []))
        for task in task_configs:
            all_seed_results[task]     = progress.get("results", {}).get(task, [])
            all_seed_predictions[task] = progress.get("predictions", {}).get(task, [])
            all_seed_labels[task]      = progress.get("labels", {}).get(task, [])
        if completed:
            print(f"\nResuming: {len(completed)} task+seed combos already done.")

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"  Seed {seed_idx+1}/{len(SEEDS)} (seed={seed})")
    print(f"{'='*60}")
    set_seed(seed)

    # Split data: 80/10/10
    train_data, val_data, test_data = {}, {}, {}
    for task_name, samples in tasks_data.items():
        shuffled = samples.copy()
        random.shuffle(shuffled)
        n  = len(shuffled)
        s1, s2 = int(0.8 * n), int(0.9 * n)
        train_data[task_name] = shuffled[:s1]
        val_data[task_name]   = shuffled[s1:s2]
        test_data[task_name]  = shuffled[s2:]

    # Train each task independently
    for task_name, num_classes in task_configs.items():
        combo_key = f"{task_name}_seed{seed}"
        if combo_key in completed:
            print(f"\n--- Skipping: {task_name} (seed={seed}) - already done ---")
            continue

        print(f"\n--- Training: {task_name} (seed={seed}) ---")
        print(f"  Train: {len(train_data[task_name])}, "
              f"Val: {len(val_data[task_name])}, "
              f"Test: {len(test_data[task_name])}")

        test_metrics, preds, labels = train_single_task(
            task_name, num_classes,
            train_data[task_name], val_data[task_name], test_data[task_name],
            tokenizer, MODEL_NAME, device,
            BATCH_SIZE, NUM_EPOCHS, MAX_LENGTH, seed=seed,
        )

        all_seed_results[task_name].append(test_metrics)
        all_seed_predictions[task_name].append([int(p) for p in preds])
        all_seed_labels[task_name].append([int(l) for l in labels])
        completed.add(combo_key)

        print(f"  Test: Acc={test_metrics['accuracy']:.4f}, "
              f"P={test_metrics['precision']:.4f}, "
              f"R={test_metrics['recall']:.4f}, "
              f"F1={test_metrics['f1']:.4f}")

        # Save progress after each task+seed
        with open(progress_path, 'w') as f:
            json.dump({
                'completed': list(completed),
                'results': all_seed_results,
                'predictions': all_seed_predictions,
                'labels': all_seed_labels,
            }, f, indent=2)
        print(f"  Progress saved ({len(completed)}/{len(SEEDS) * len(task_configs)} done)")

In [ ]:
# ── Aggregated results ─────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f"  Aggregated Results (mean +/- std, {len(SEEDS)} seeds)")
print(f"{'='*60}")

aggregated = {}
for task in task_configs:
    agg = {}
    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        values = [m[metric] for m in all_seed_results[task]]
        agg[metric] = {'mean': float(np.mean(values)), 'std': float(np.std(values)),
                       'per_seed': [float(v) for v in values]}
    aggregated[task] = agg
    print(f"\n  {task}:")
    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        print(f"    {metric:>10s}: {agg[metric]['mean']:.4f} +/- {agg[metric]['std']:.4f}")

with open(os.path.join(RESULTS_DIR, 'aggregated_results.json'), 'w') as f:
    json.dump(aggregated, f, indent=2)

# Per-class emotion metrics
best_idx  = int(np.argmax([m['f1'] for m in all_seed_results['emotion']]))
per_class = compute_per_class_metrics(
    all_seed_predictions['emotion'][best_idx],
    all_seed_labels['emotion'][best_idx],
    EMOTION_CLASSES
)
print(f"\n  Per-Class Emotion (best seed: {SEEDS[best_idx]}):")
print(per_class['report_str'])

per_class_save = dict(per_class['report_dict'])
per_class_save['confusion_matrix'] = per_class['confusion_matrix']
with open(os.path.join(RESULTS_DIR, 'emotion_per_class_metrics.json'), 'w') as f:
    json.dump(per_class_save, f, indent=2)

print("\n  Done!")